<a href="https://colab.research.google.com/github/240266-alt/AIML-verification/blob/main/Copy_of_ucs_and_a_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

In [ ]:
df = files.upload()

In [ ]:
import pandas as pd
import heapq

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df=pd.read_csv("Clean_Dataset.csv")
# Display basic information about the dataset



print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# Remove unnecessary columns if they exist
df.drop(
    columns=["Unnamed: 0", "flight"],
    errors="ignore",
    inplace=True
)

print("\n" + "=" * 50)
print("LINEAR REGRESSION")
print("=" * 50)


# Encode categorical columns
categorical_columns = df.select_dtypes(
    include=["object"]
).columns

label_encoders = {}

for column in categorical_columns:

    encoder = LabelEncoder()

    df[column] = encoder.fit_transform(
        df[column].astype(str)
    )

    label_encoders[column] = encoder


# Separate features and target
X = df.drop("price", axis=1)
y = df["price"]


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# Create model
model = LinearRegression()


# Train model
model.fit(X_train, y_train)


# Prediction
y_pred = model.predict(X_test)


# Evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)


print("Mean Squared Error:", mse)
print("R2 Score:", r2)


# Show sample predictions
results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

print("\nSample Predictions:")
print(results.head(10))


# ============================================================
# 3. BUILD GRAPH FOR UCS AND A*
# ============================================================

print("\n" + "=" * 50)
print("BUILDING FLIGHT GRAPH")
print("=" * 50)


# Load original dataset again because the previous dataframe
# has encoded city names.
graph_df = pd.read_csv("Clean_Dataset.csv")


# Remove unnecessary column
graph_df.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
    inplace=True
)


# Create graph
graph = {}


for _, row in graph_df.iterrows():

    source = row["source_city"]
    destination = row["destination_city"]

    # Duration is used as the edge cost
    cost = float(row["duration"])

    if source not in graph:
        graph[source] = []

    graph[source].append(
        (destination, cost)
    )


# ============================================================
# 4. UNIFORM COST SEARCH
# ============================================================

def uniform_cost_search(graph, start, goal):

    # Priority queue:
    # (total_cost, current_node, path)

    priority_queue = [
        (0, start, [])
    ]

    visited = set()

    nodes_expanded = 0


    while priority_queue:

        cost, current, path = heapq.heappop(
            priority_queue
        )

        if current in visited:
            continue

        visited.add(current)

        nodes_expanded += 1

        path = path + [current]


        # Goal reached
        if current == goal:

            return (
                path,
                cost,
                nodes_expanded
            )


        # Explore neighbours
        for neighbor, edge_cost in graph.get(
            current, []
        ):

            if neighbor not in visited:

                new_cost = cost + edge_cost

                heapq.heappush(
                    priority_queue,
                    (
                        new_cost,
                        neighbor,
                        path
                    )
                )


    return None, float("inf"), nodes_expanded


# ============================================================
# 5. HEURISTIC FOR A*
# ============================================================


heuristic_values = {

    "Delhi": 3.0,
    "Mumbai": 0.0,
    "Bangalore": 2.0,
    "Hyderabad": 1.5,
    "Chennai": 2.0,
    "Kolkata": 3.5
}


def heuristic(node, goal):

    # If goal is Mumbai, use the values above.
    # Otherwise return 0 for unknown cases.

    return heuristic_values.get(node, 0)


# ============================================================
# 6. A* SEARCH
# ============================================================

def a_star_search(graph, start, goal):

    # Priority queue:
    # (f, g, current_node, path)

    # f(n) = g(n) + h(n)

    initial_g = 0

    initial_f = (
        initial_g +
        heuristic(start, goal)
    )


    priority_queue = [
        (
            initial_f,
            initial_g,
            start,
            []
        )
    ]

    visited = set()

    nodes_expanded = 0


    while priority_queue:

        f, g, current, path = heapq.heappop(
            priority_queue
        )

        if current in visited:
            continue

        visited.add(current)

        nodes_expanded += 1

        path = path + [current]


        # Goal reached
        if current == goal:

            return (
                path,
                g,
                nodes_expanded
            )


        # Explore neighbours
        for neighbor, edge_cost in graph.get(
            current, []
        ):

            if neighbor not in visited:

                new_g = g + edge_cost

                h = heuristic(
                    neighbor,
                    goal
                )

                new_f = new_g + h


                heapq.heappush(
                    priority_queue,
                    (
                        new_f,
                        new_g,
                        neighbor,
                        path
                    )
                )


    return None, float("inf"), nodes_expanded


# ============================================================
# 7. RUN UCS
# ============================================================

start_city = "Delhi"
goal_city = "Mumbai"


ucs_path, ucs_cost, ucs_nodes = uniform_cost_search(
    graph,
    start_city,
    goal_city
)


print("\n" + "=" * 50)
print("UNIFORM COST SEARCH")
print("=" * 50)

print("Start:", start_city)
print("Goal:", goal_city)

print("Optimal Path:", ucs_path)

print("Total Cost / Duration:", ucs_cost)

print("Nodes Expanded:", ucs_nodes)


# ============================================================
# 8. RUN A*
# ============================================================

astar_path, astar_cost, astar_nodes = a_star_search(
    graph,
    start_city,
    goal_city
)


print("\n" + "=" * 50)
print("A* SEARCH")
print("=" * 50)

print("Start:", start_city)
print("Goal:", goal_city)

print("Optimal Path:", astar_path)

print("Total Cost / Duration:", astar_cost)

print("Nodes Expanded:", astar_nodes)


# ============================================================
# 9. COMPARISON
# ============================================================

print("\n" + "=" * 50)
print("UCS vs A* COMPARISON")
print("=" * 50)

print(
    "\nUCS Path:",
    ucs_path
)

print(
    "UCS Cost:",
    ucs_cost
)

print(
    "UCS Nodes Expanded:",
    ucs_nodes
)

print(
    "\nA* Path:",
    astar_path
)

print(
    "A* Cost:",
    astar_cost
)

print(
    "A* Nodes Expanded:",
    astar_nodes
)

Dataset Shape: (300153, 12)

Columns:
['Unnamed: 0', 'airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class', 'duration', 'days_left', 'price']

LINEAR REGRESSION
Mean Squared Error: 49200540.293721996
R2 Score: 0.904554350718816

Sample Predictions:
   Actual Price  Predicted Price
0          7366      4585.880510
1         64831     52842.077073
2          6195      7877.980184
3         60160     56120.138579
4          6578      6663.204177
5          4555      6829.586916
6         23838     48754.677406
7          3860      4020.662202
8         32230     49953.512219
9         76841     58193.510615

BUILDING FLIGHT GRAPH

UNIFORM COST SEARCH
Start: Delhi
Goal: Mumbai
Optimal Path: ['Delhi', 'Mumbai']
Total Cost / Duration: 2.0
Nodes Expanded: 3

A* SEARCH
Start: Delhi
Goal: Mumbai
Optimal Path: ['Delhi', 'Mumbai']
Total Cost / Duration: 2.0
Nodes Expanded: 2

UCS vs A* COMPARISON

UCS Path: ['Delhi', 'Mumbai']
UCS Cost: 2.0
UCS